# 330 — Anatomical parcellation classification (Yeo-7 & Yeo-17)

Decodes **which Yeo functional network** an electrode sits in, from its response profile
**concatenated across all three conditions** (`audio ⊕ picture ⊕ reading`). Run for each of
the 4 feature variants × {Yeo-7, Yeo-17} × each classifier = 16 experiments.

The classes are imbalanced (some networks have far more electrodes than others), so
**balanced accuracy** and **macro-F1** are the headline metrics and `class_weight='balanced'`
is used throughout. Same nested-GroupKFold-by-patient protocol as 320 — so a network is only
"decodable" if it generalises across patients, not because a couple of patients dominate it.

See `390_results.ipynb` for the figures and how to read them.


In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

# ---------------- config knobs ----------------
YEO_NETWORKS = (7, 17)
CLASSIFIERS  = ('logreg', 'rf')
OUTER_SPLITS = 5
INNER_SPLITS = 3
N_PERM       = 200
N_BOOT       = 1000
RANDOM_STATE = 42
VARIANTS_TO_RUN = C.VARIANTS   # subset for a fast pass (see 320)
print('yeo:', YEO_NETWORKS, '| classifiers:', CLASSIFIERS,
      '| variants:', VARIANTS_TO_RUN, '| n_perm:', N_PERM)


yeo: (7, 17) | classifiers: ('logreg', 'rf') | variants: ('full_300', 'hg_300', 'full_30', 'hg_30', 'm101_300', 'm101_30', 'full_300_rn', 'full_30_rn') | n_perm: 200


## Run all parcellation experiments
2 Yeo granularities × 8 variants × 2 classifiers = 32 runs (fewer if you subset
`VARIANTS_TO_RUN`), saved under
`outputs/classification/parcellation_yeo{7,17}/<variant>/<classifier>/runs/<id>/`.

The per-class **response profile** is the full ERSP of each network, **concatenated
across audio+picture+reading** (a spectrogram triptych, not a single line), with a
grey stim→response boundary per condition.


In [ ]:
manifests = []
for n_net in YEO_NETWORKS:
    key = f'yeo{n_net}'
    # full-spectrum, 3 conditions concatenated -> the per-class ERSP triptych
    Xf, yf, gf, mf, cf = C.load_arrays('parcellation', key, 'full_300')
    profile = {'X': Xf, 'n_time': 300, 'n_cond': 3, 'cond_names': list(C.CONDITIONS)}
    for v in VARIANTS_TO_RUN:
        X, y, groups, meta, cols = C.load_arrays('parcellation', key, v)
        for clf in CLASSIFIERS:
            m = C.run_experiment(f'parcellation_{key}', v, clf, X, y, groups, cols, meta,
                                 n_networks=n_net, profile=profile,
                                 outer_splits=OUTER_SPLITS, inner_splits=INNER_SPLITS,
                                 n_perm=N_PERM, n_boot=N_BOOT, random_state=RANDOM_STATE)
            manifests.append(m)
print('\ndone:', len(manifests), 'runs')



=== parcellation_yeo7 | full_300 | logreg -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\classification\parcellation_yeo7\full_300\logreg\runs\20260609_121312 ===
  outer fold 1/5: best={'clf__C': 0.1}
  outer fold 2/5: best={'clf__C': 0.01}
  outer fold 3/5: best={'clf__C': 0.01}
  outer fold 4/5: best={'clf__C': 0.01}
  outer fold 5/5: best={'clf__C': 0.1}
  permutation 40/200
  permutation 80/200
  permutation 120/200
  permutation 160/200
  permutation 200/200
[feature_importance:logreg] best={'clf__C': 0.01}
  bal_acc=0.307 (chance 0.143)  macro_F1=0.313  p=0.0050

=== parcellation_yeo7 | full_300 | rf -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\classification\parcellation_yeo7\full_300\rf\runs\20260609_130158 ===
  outer fold 1/5: best={'clf__max_depth': 8, 'clf__min_samples_leaf': 3}
  outer fold 2/5: best={'clf__max_depth': 8, 'clf__min_samples_leaf': 3}
  outer fold 3/5:

## Summary table


In [ ]:
df = C.list_runs()
df = df[df.task.str.startswith('parcellation')]
df[['task', 'variant', 'classifier', 'balanced_accuracy', 'chance_level',
    'macro_f1', 'permutation_p']].round(4)
